<a href="https://colab.research.google.com/github/Lahari1421/Calculator.Codsoft/blob/main/MOVIERECOMMENDATIONSYS-AICTE-AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install pandas scikit-learn numpy


In [3]:
!wget https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -q ml-100k.zip


--2025-05-02 06:35:35--  https://files.grouplens.org/datasets/movielens/ml-100k.zip
Resolving files.grouplens.org (files.grouplens.org)... 128.101.65.152
Connecting to files.grouplens.org (files.grouplens.org)|128.101.65.152|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4924029 (4.7M) [application/zip]
Saving to: ‘ml-100k.zip’

ml-100k.zip         100%[===================>]   4.70M  16.0MB/s    in 0.3s    

2025-05-02 06:35:36 (16.0 MB/s) - ‘ml-100k.zip’ saved [4924029/4924029]



In [6]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# Load ratings data
columns = ['user_id', 'movie_id', 'rating', 'timestamp']
ratings = pd.read_csv('ml-100k/u.data', sep='\t', names=columns)

# Load movie titles
movie_columns = ['movie_id', 'movie_title'] + [str(i) for i in range(22)]
movies = pd.read_csv('ml-100k/u.item', sep='|', encoding='latin-1', names=movie_columns, usecols=[0, 1])

# Merge datasets
data = pd.merge(ratings, movies, on='movie_id')

# Create user-movie rating matrix
user_movie_matrix = data.pivot_table(index='user_id', columns='movie_title', values='rating')

# Fill missing values with 0
user_movie_matrix_filled = user_movie_matrix.fillna(0)

# Compute user similarity matrix using cosine similarity
user_similarity = cosine_similarity(user_movie_matrix_filled)

# Convert similarity matrix to DataFrame
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

# Recommend movies function
def recommend_movies(user_id, num_recommendations=5):
    if user_id not in user_similarity_df.index:
        return "User ID not found."

    sim_scores = user_similarity_df[user_id]
    sim_scores = sim_scores.drop(user_id)
    similar_users = sim_scores.sort_values(ascending=False).head(5).index

    user_movies = user_movie_matrix.loc[user_id]
    user_seen_movies = user_movies[user_movies.notnull()].index

    recommended_movies = pd.Series(dtype='float64')

    for similar_user in similar_users:
        similar_user_ratings = user_movie_matrix.loc[similar_user]
        similar_user_recommendations = similar_user_ratings.drop(user_seen_movies)
        recommended_movies = pd.concat([recommended_movies, similar_user_recommendations])

    recommended_movies = recommended_movies.groupby(recommended_movies.index).mean()
    recommended_movies = recommended_movies.dropna()

    top_recommendations = recommended_movies.sort_values(ascending=False).head(num_recommendations)

    return top_recommendations


In [7]:
# Example Usage
print("Recommended Movies for User 5:\n")
print(recommend_movies(5, 5))


Recommended Movies for User 5:

Sense and Sensibility (1995)        5.0
Executive Decision (1996)           5.0
Reservoir Dogs (1992)               5.0
Secret of Roan Inish, The (1994)    5.0
Dante's Peak (1997)                 5.0
dtype: float64


In [8]:
print(recommend_movies(10, 5))
print(recommend_movies(50, 10))


Misérables, Les (1995)          5.0
In the Company of Men (1997)    5.0
Boys, Les (1997)                5.0
Braindead (1992)                5.0
Thin Blue Line, The (1988)      5.0
dtype: float64
Emma (1996)                        5.000000
Courage Under Fire (1996)          5.000000
Godfather, The (1972)              4.666667
Star Wars (1977)                   4.400000
Return of the Jedi (1983)          4.333333
Mighty Aphrodite (1995)            4.333333
Welcome to the Dollhouse (1995)    4.250000
Donnie Brasco (1997)               4.250000
Bound (1996)                       4.250000
Bottle Rocket (1996)               4.000000
dtype: float64


In [9]:
user_input = int(input("Enter User ID (1-943): "))
print(recommend_movies(user_input, 5))


Enter User ID (1-943): 786
101 Dalmatians (1996)           5.0
12 Angry Men (1957)             5.0
Time to Kill, A (1996)          5.0
To Kill a Mockingbird (1962)    5.0
True Crime (1995)               5.0
dtype: float64


In [10]:
recommendations = recommend_movies(5, 10)
recommendations.to_csv("user5_recommendations.csv")


In [11]:
top_movies = data.groupby('movie_title')['rating'].mean().sort_values(ascending=False).head(10)
print(top_movies)


movie_title
Aiqing wansui (1994)                                 5.0
Entertaining Angels: The Dorothy Day Story (1996)    5.0
Santa with Muscles (1996)                            5.0
Prefontaine (1997)                                   5.0
They Made Me a Criminal (1939)                       5.0
Saint of Fort Washington, The (1993)                 5.0
Great Day in Harlem, A (1994)                        5.0
Star Kid (1997)                                      5.0
Marlene Dietrich: Shadow and Light (1996)            5.0
Someone Else's America (1995)                        5.0
Name: rating, dtype: float64
